In [ ]:
%pip install pgmpy -q

In [ ]:
import numpy as np
from scipy.stats import binom
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

# in versiunile noi de pgmpy clasa se numeste DiscreteBayesianNetwork, in cele vechi BayesianNetwork
try:
    from pgmpy.models import DiscreteBayesianNetwork as Retea
except ImportError:
    from pgmpy.models import BayesianNetwork as Retea

**Ex. 1 (Monty Hall)** Sunt trei usi, iar in spatele uneia se afla un premiu (in spatele celorlalte doua sunt capre). Concurentul alege o usa. Prezentatorul, care stie unde este premiul, deschide **o alta usa, mereu fara premiu**, apoi il intreaba pe concurent daca vrea sa isi schimbe alegerea. Merita sa schimbe? Da: la schimbare probabilitatea de castig este $2/3$, iar daca ramane la prima usa este $1/3$.

Modelul din pgmpy are trei variabile: `Guest` (usa aleasa de concurent) si `Prize` (usa cu premiul), ambele uniforme pe $\{A, B, C\}$, si `Host` (usa deschisa de prezentator), care depinde de amandoua. Prezentatorul nu poate deschide nici usa aleasa, nici usa cu premiul, iar daca are doua usi permise o alege la intamplare.

Pentru $Guest = A$ si $Host = C$, cu Bayes:

$$P(Prize = A \mid G = A, H = C) = \frac{\frac13 \cdot \frac12}{\frac13 \cdot \frac12 + \frac13 \cdot 1 + \frac13 \cdot 0} = \frac13, \qquad P(Prize = B \mid G = A, H = C) = \frac23$$

In [ ]:
# ex. 1: monty hall
usi = ["A", "B", "C"]

# concurentul alege o usa uniform, iar premiul e si el pus uniform
cpd_guest = TabularCPD("Guest", 3, [[1 / 3], [1 / 3], [1 / 3]], state_names={"Guest": usi})
cpd_prize = TabularCPD("Prize", 3, [[1 / 3], [1 / 3], [1 / 3]], state_names={"Prize": usi})

# P(Host | Guest, Prize): construim cate o coloana pentru fiecare pereche (guest, prize)
coloane = []
for guest in usi:
    for prize in usi:
        # prezentatorul poate deschide doar o usa care nu e aleasa si nu are premiul
        permise = [u for u in usi if u != guest and u != prize]
        # probabilitate egala intre usile permise, 0 pentru celelalte
        coloane.append([1 / len(permise) if u in permise else 0 for u in usi])
# transpunem ca randurile sa fie usa deschisa si coloanele perechile (guest, prize)
valori_host = np.array(coloane).T

cpd_host = TabularCPD("Host", 3, valori_host,
                      evidence=["Guest", "Prize"], evidence_card=[3, 3],
                      state_names={"Host": usi, "Guest": usi, "Prize": usi})

monty = Retea([("Guest", "Host"), ("Prize", "Host")])
monty.add_cpds(cpd_guest, cpd_prize, cpd_host)
assert monty.check_model()

# concurentul alege A, iar prezentatorul deschide C: unde e premiul?
infer_monty = VariableElimination(monty)
print(infer_monty.query(["Prize"], evidence={"Guest": "A", "Host": "C"}))
# A are 1/3 si B are 2/3, deci concurentului ii convine sa schimbe usa

In [ ]:
# ex. 2 a): modelul probabilist (variabile binare, starile sunt 0 si 1)
# structura: B -> T, B -> X, B -> D si T -> D
boala = Retea([("B", "T"), ("B", "X"), ("B", "D"), ("T", "D")])

# P(B): randul 0 este B=0, randul 1 este B=1
cpd_B = TabularCPD("B", 2, [[0.9], [0.1]])

# P(T | B): coloanele sunt B=0 si B=1
cpd_T = TabularCPD("T", 2,
                   [[0.7, 0.2],    # T=0
                    [0.3, 0.8]],   # T=1
                   evidence=["B"], evidence_card=[2])

# P(X | B)
cpd_X = TabularCPD("X", 2,
                   [[0.9, 0.1],    # X=0
                    [0.1, 0.9]],   # X=1
                   evidence=["B"], evidence_card=[2])

# P(D | B, T): coloanele sunt (B=0,T=0), (B=0,T=1), (B=1,T=0), (B=1,T=1)
cpd_D = TabularCPD("D", 2,
                   [[0.9, 0.5, 0.6, 0.1],   # D=0
                    [0.1, 0.5, 0.4, 0.9]],  # D=1
                   evidence=["B", "T"], evidence_card=[2, 2])

boala.add_cpds(cpd_B, cpd_T, cpd_X, cpd_D)
assert boala.check_model()

infer_boala = VariableElimination(boala)

# ex. 2 b): P(B=1 | T=1, D=1)
print(infer_boala.query(["B"], evidence={"T": 1, "D": 1}))

# ex. 2 c): P(X=1 | B=0, T=1, D=1)
# X depinde doar de B, asa ca rezultatul trebuie sa fie P(X=1 | B=0) = 0.1
print(infer_boala.query(["X"], evidence={"B": 0, "T": 1, "D": 1}))

**Ex. 2 d)** Din structura retelei, $P(B, T, X, D) = P(B)\,P(T \mid B)\,P(X \mid B)\,P(D \mid B, T)$.

**b)** Variabila $X$ nu apare in evidenta, deci se elimina prin sumare (suma lui $P(X \mid B)$ peste $X$ este 1). Cu Bayes:

$$P(B = 1 \mid T = 1, D = 1) = \frac{P(B=1)\,P(T=1 \mid B=1)\,P(D=1 \mid B=1, T=1)}{\sum_{b} P(B=b)\,P(T=1 \mid B=b)\,P(D=1 \mid B=b, T=1)}$$

$$= \frac{0.1 \cdot 0.8 \cdot 0.9}{0.1 \cdot 0.8 \cdot 0.9 + 0.9 \cdot 0.3 \cdot 0.5} = \frac{0.072}{0.072 + 0.135} = \frac{0.072}{0.207} = \frac{8}{23} \approx 0.348$$

**c)** Din retea, $X \perp \{T, D\} \mid B$: tot drumul de la $X$ la $T$ si $D$ trece prin $B$ ($X \leftarrow B \rightarrow T$ si $X \leftarrow B \rightarrow D$), care este observat si il blocheaza. Deci:

$$P(X = 1 \mid B = 0, T = 1, D = 1) = \frac{P(B{=}0)\,P(T{=}1 \mid B{=}0)\,P(X{=}1 \mid B{=}0)\,P(D{=}1 \mid B{=}0, T{=}1)}{P(B{=}0)\,P(T{=}1 \mid B{=}0)\,P(D{=}1 \mid B{=}0, T{=}1)} = P(X = 1 \mid B = 0) = 0.1$$

In [ ]:
# ex. 3 a): simulam 10000 de jocuri
rng = np.random.default_rng()
n_jocuri = 10000

incepe = rng.integers(0, 2, size=n_jocuri)    # moneda normala: 0 = J0 incepe, 1 = J1 incepe
n = rng.integers(1, 7, size=n_jocuri)         # zarul din prima runda (1..6)

# in a doua runda arunca celalalt jucator: daca incepe J0, a doua runda e a lui J1,
# care are moneda masluita (stema cu probabilitatea 4/7); altfel moneda e normala
p_stema = np.where(incepe == 0, 4 / 7, 0.5)
m = rng.binomial(2 * n, p_stema)              # numarul de steme din 2n aruncari

# jucatorul din prima runda castiga daca n >= m, altfel castiga celalalt
castigator = np.where(n >= m, incepe, 1 - incepe)

p_J0 = np.mean(castigator == 0)
p_J1 = np.mean(castigator == 1)
print(f"J0 castiga in {p_J0:.3f} din jocuri, J1 in {p_J1:.3f}")
print("are sanse mai mari:", "J0" if p_J0 > p_J1 else "J1")

In [ ]:
# ex. 3 b): reteaua bayesiana
# S = cine incepe, N = rezultatul zarului, M = numarul de steme din a doua runda
# M depinde de N (numarul de aruncari e 2N) si de S (de la S depinde moneda folosita)
joc = Retea([("S", "M"), ("N", "M")])

jucatori = ["J0", "J1"]
cpd_S = TabularCPD("S", 2, [[0.5], [0.5]], state_names={"S": jucatori})
cpd_N = TabularCPD("N", 6, [[1 / 6]] * 6, state_names={"N": list(range(1, 7))})

# P(M | S, N): coloanele merg prin (S=J0, N=1..6), apoi (S=J1, N=1..6)
coloane_M = []
for s in jucatori:
    # daca incepe J0, a doua runda o joaca J1 cu moneda masluita
    p = 4 / 7 if s == "J0" else 0.5
    for nr in range(1, 7):
        # M ~ binomiala(2N, p), pmf da 0 pentru valorile imposibile (M > 2N)
        coloane_M.append([binom.pmf(m, 2 * nr, p) for m in range(13)])
valori_M = np.array(coloane_M).T

cpd_M = TabularCPD("M", 13, valori_M,
                   evidence=["S", "N"], evidence_card=[2, 6],
                   state_names={"M": list(range(13)), "S": jucatori, "N": list(range(1, 7))})

joc.add_cpds(cpd_S, cpd_N, cpd_M)
assert joc.check_model()

In [ ]:
# ex. 3 c): cine a inceput jocul, stiind ca in a doua runda s-a obtinut o singura stema (M=1)?
infer_joc = VariableElimination(joc)
posterior_S = infer_joc.query(["S"], evidence={"M": 1})
print(posterior_S)
# jucatorul cu probabilitatea mai mare e cel mai probabil sa fi inceput jocul